<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/The_Great_Pendulum_Balance_Off_PID_vs_LQR_vs_MPC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inverted Pendulum Control Benchmark: PID vs. LQR vs. MPC

## Overview
This notebook implements a high-fidelity physics simulation of an inverted pendulum on a cart. It evaluates three distinct control architectures—Proportional-Integral-Derivative (PID), Linear-Quadratic Regulator (LQR), and Model Predictive Control (MPC)—on their ability to maintain equilibrium and recover from significant external disturbances.

## System Dynamics
The system is modeled using nonlinear equations of motion integrated via the Fourth-order Runge-Kutta (RK4) method. The linearized state-space representation is used for controller synthesis, defined by the state vector:

$$x = [p, v, \theta, \omega]^T$$

Where:
* **p**: Cart position (m)
* **v**: Cart velocity (m/s)
* **̘**: Pole angle (rad), where 0 is the upright equilibrium.
* **̑**: Pole angular velocity (rad/s)

## Control Methodologies

### 1. PID Controller
A standard feedback loop using angle and angular rate. It serves as a baseline, utilizing an anti-windup integrator. Notably, this implementation omits cart position feedback to demonstrate intentional drift characteristics compared to state-space methods.

### 2. Linear-Quadratic Regulator (LQR)
An optimal control strategy that minimizes a quadratic cost function involving state error and control effort. The gain matrix K is derived by solving the continuous-time Algebraic Riccati Equation (ARE).

### 3. Model Predictive Control (MPC)
A predictive strategy that solves an optimization problem over a finite time horizon (25 steps) at each sampling interval. It utilizes a discrete-time model to anticipate future system behavior and optimize current control actions.

## Simulation and Evaluation
The simulation runs for 12 seconds with a 5ms timestep. A significant angular velocity impulse is applied at the 3-second mark to test the robustness of each controller. Performance is quantified based on:
* **Peak Deviation**: Maximum angle reached after disturbance.
* **Recovery Time**: Time required to return the pole to within 1.5 degrees of vertical.
* **Control Energy**: The integral of the squared control force over time.

## Requirements
* NumPy
* Matplotlib
* SciPy
* FFmpeg (for video rendering)

In [4]:
"""
The Great Balance-Off: PID vs. LQR vs. MPC
===========================================
Author   : Mugambi Ndwiga
Instagram: @craftsandengineering

Three controllers compete on identical inverted-pendulum-on-cart systems.
After a steady-state phase, an angular-velocity impulse hits all three
simultaneously. Performance is ranked on peak pole angle, recovery time,
and cumulative control energy.

Controllers
-----------
PID : angle + rate feedback only (no cart position), anti-windup integrator.
      Gains set to 50 % of the LQR equivalents — cart drift is intentional
      and is labelled in the video.
LQR : full-state continuous-time regulator via algebraic Riccati equation.
MPC : unconstrained condensed QP, 25-step prediction horizon, Ts = 5 ms.

Rendering notes
---------------
- Camera follows each cart so the pendulum fills the panel at all times.
- Arenas occupy 88 % of frame height; plots are a compact 6 % strip below.
- 2x slow motion: SLOW = 2 halves the real-time playback speed.
- DPI = 90, T_SIM = 12 s  =>  ~370 frames, renders in ~60 s.
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch, Arc
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.lines import Line2D
from scipy.linalg import solve_continuous_are, expm
import warnings, os

warnings.filterwarnings("ignore")

# =============================================================================
# 1.  PHYSICAL PARAMETERS
# =============================================================================
M_CART = 1.0    # cart mass                [kg]
M_POLE = 0.2    # pole mass (lumped at tip) [kg]
L_HALF = 0.6    # pivot-to-tip length       [m]
G      = 9.81   # gravitational accel.      [m/s^2]
B_CART = 0.05   # cart viscous damping      [N.s/m]
B_POLE = 0.002  # pole pivot damping        [N.m.s/rad]

DT     = 0.005  # integration timestep      [s]
T_SIM  = 12.0   # simulation duration       [s]   (rec #10)
N      = int(T_SIM / DT)

DIST_T      = 3.0   # disturbance onset     [s]
DIST_ANGVEL = 5.0   # angular velocity kick [rad/s]

# Video timing (rec #9, #10)
SLOW  = 2           # slow-motion factor
FPS   = 30
EVERY = max(1, int(round(1.0 / (DT * FPS * SLOW))))

# =============================================================================
# 2.  STATE-SPACE  (linearised about upright equilibrium)
#     x = [cart_pos, cart_vel, pole_angle, pole_rate]
# =============================================================================
A = np.array([
    [0,  1,                          0,                                 0],
    [0, -B_CART/M_CART,              M_POLE*G/M_CART,                   0],
    [0,  0,                          0,                                 1],
    [0,  B_CART/(M_CART*L_HALF),    -(M_CART+M_POLE)*G/(M_CART*L_HALF), 0],
])
B = np.array([[0], [1.0/M_CART], [0], [-1.0/(M_CART*L_HALF)]])

# =============================================================================
# 3.  CONTROLLER DESIGNS
# =============================================================================

# --- LQR --------------------------------------------------------------------
Q_lqr = np.diag([1.0, 0.5, 200.0, 20.0])
R_lqr = np.array([[0.01]])
K_lqr = (1.0/R_lqr[0,0]) * B.T @ solve_continuous_are(A, B, Q_lqr, R_lqr)

# --- MPC (discrete, zero-order hold) ----------------------------------------
def c2d(Ac, Bc, dt):
    n  = Ac.shape[0]
    em = expm(np.block([[Ac, Bc], [np.zeros((1, n+1))]]) * dt)
    return em[:n, :n], em[:n, n:]

Ad, Bd = c2d(A, B, DT)
N_HOR  = 25
Q_mpc  = np.diag([0.8, 0.4, 150.0, 18.0])
R_mpc  = np.array([[0.08]])

def _build_mpc(Ad, Bd, Q, R, Nh):
    n, p = Ad.shape[0], Bd.shape[1]
    Phi = np.zeros((n*Nh, n)); Gam = np.zeros((n*Nh, p*Nh))
    Ak  = np.eye(n)
    for i in range(Nh):
        Ak = Ad @ Ak; Phi[i*n:(i+1)*n] = Ak
        for j in range(i+1):
            Gam[i*n:(i+1)*n, j*p:(j+1)*p] = np.linalg.matrix_power(Ad, i-j) @ Bd
    Qb = np.kron(np.eye(Nh), Q); Rb = np.kron(np.eye(Nh), R)
    H  = Gam.T@Qb@Gam + Rb; F = Gam.T@Qb@Phi
    return np.linalg.inv(H), F

_Hi, _Fm = _build_mpc(Ad, Bd, Q_mpc, R_mpc, N_HOR)

def mpc_control(x):
    return float(np.clip(-(_Hi @ _Fm @ x.reshape(-1,1))[0,0], -60., 60.))

# --- PID (rec #2, #14, #18) -------------------------------------------------
KP_PID = float(K_lqr[0,2]) * 0.50   # 50 % of LQR angle gain
KD_PID = float(K_lqr[0,3]) * 0.50   # 50 % of LQR rate gain
KI_PID = 3.0
U_MAX  = 60.0

def pid_step(th, thd, itg):
    u_raw = -(KP_PID*th + KD_PID*thd + KI_PID*itg)
    u_sat = float(np.clip(u_raw, -U_MAX, U_MAX))
    return u_sat, itg + (th + 0.5*(u_sat - u_raw)/(KI_PID+1e-12)) * DT

# =============================================================================
# 4.  NONLINEAR DYNAMICS  (RK4)
# =============================================================================
def rk4(s, u, dt):
    def f(s):
        _, xd, th, thd = s
        c, si = np.cos(th), np.sin(th)
        dn    = M_CART + M_POLE*si**2
        xdd   = (u - B_CART*xd + M_POLE*si*(L_HALF*thd**2 + G*c)) / dn
        thdd  = (-u*c - M_POLE*L_HALF*thd**2*c*si
                 - (M_CART+M_POLE)*G*si - B_POLE*thd) / (L_HALF*dn)
        return np.array([xd, xdd, thd, thdd])
    k1=f(s); k2=f(s+.5*dt*k1); k3=f(s+.5*dt*k2); k4=f(s+dt*k3)
    return s + dt/6*(k1 + 2*k2 + 2*k3 + k4)

# =============================================================================
# 5.  SIMULATION
# =============================================================================
CTRL_NAMES = ("PID", "LQR", "MPC")

def run_all():
    out = {}
    for ctrl in CTRL_NAMES:
        sts = np.zeros((N,4)); frc = np.zeros(N)
        s   = np.zeros(4); itg = 0.0
        for k in range(N):
            t = k*DT
            if abs(t - DIST_T) < DT/2:
                s[3] += DIST_ANGVEL          # angular velocity impulse
            if   ctrl == "PID": u, itg = pid_step(s[2], s[3], itg)
            elif ctrl == "LQR": u = float(np.clip(-(K_lqr@s)[0], -U_MAX, U_MAX))
            else:               u = mpc_control(s)
            frc[k]=u; sts[k]=s
            s = rk4(s, u, DT)
            s[0] = np.clip(s[0], -12., 12.)  # runaway guard only (rec #2)
        out[ctrl] = dict(states=sts, forces=frc)
    return out

print("Simulating ...")
SIM      = run_all()
time_arr = np.arange(N) * DT

# =============================================================================
# 6.  METRICS
# =============================================================================
di = int(DIST_T / DT)

metrics = {}
for ctrl in CTRL_NAMES:
    th    = SIM[ctrl]["states"][:,2]
    fc    = SIM[ctrl]["forces"]
    peak  = float(np.max(np.abs(th[di:])) * 180/np.pi)
    nrg   = float(np.sum(fc[di:]**2) * DT)
    rec_i = np.where(np.abs(th[di+20:])*180/np.pi < 1.5)[0]
    recov = float(rec_i[0]*DT) if len(rec_i) else T_SIM - DIST_T
    metrics[ctrl] = dict(peak_deg=peak, energy=nrg, recovery_s=recov)

def rank_asc(key):
    order = sorted(CTRL_NAMES, key=lambda c: metrics[c][key])
    return {c: order.index(c)+1 for c in CTRL_NAMES}

rP = rank_asc("peak_deg"); rE = rank_asc("energy"); rR = rank_asc("recovery_s")
scores = {c: rP[c]+rE[c]+rR[c] for c in CTRL_NAMES}
WINNER = min(scores, key=lambda c: scores[c])

print(f"  Metrics : {metrics}")
print(f"  Winner  : {WINNER}  scores={scores}")

cum_nrg = {c: np.cumsum(SIM[c]["forces"]**2)*DT for c in CTRL_NAMES}
MAX_NRG = max(v[-1] for v in cum_nrg.values()) * 1.08

# =============================================================================
# 7.  STYLE
# =============================================================================
BG    = "#07071a"; PANEL = "#0b0b1e"; INSET = "#0d0d24"
BDR   = "#1c1c3c"; WHITE = "#dcdcec"; DIM   = "#40406a"
DIM2  = "#6a6a9a"; GOLD  = "#c8a800"; RED_C = "#cc3333"
AMBER = "#d4913a"; GRN   = "#55cc66"

COLS  = {"PID": "#e05c5c", "LQR": "#4ecdc4", "MPC": "#c8a800"}

# Camera viewport constants (rec #1)
CAM_HALF = 1.4     # half-width of camera window   [m]
CAM_Y0   = -0.40
CAM_Y1   =  1.90
CART_W   = 0.38
CART_H   = 0.17
POLE_VIS = L_HALF * 2.1   # visual pole length

plt.rcParams.update({
    "font.family"    : "DejaVu Sans",
    "text.color"     : WHITE,
    "axes.labelcolor": DIM2,
    "xtick.color"    : DIM2,
    "ytick.color"    : DIM2,
    "axes.facecolor" : INSET,
    "figure.facecolor": BG,
    "axes.grid"      : False,
})

# =============================================================================
# 8.  FIGURE LAYOUT  (rec #3, #4, #5, #21, #22)
#
#   Row 0   title strip            3 %
#   Row 1   three arena panels    88 %   DOMINANT
#   Row 2   phase / status strip   3 %
#   Row 3   four compact plots     6 %
# =============================================================================
fig = plt.figure(figsize=(20, 11.25), facecolor=BG)   # 16:9

outer = gridspec.GridSpec(4, 1, figure=fig,
    height_ratios=[0.03, 0.88, 0.03, 0.06], hspace=0.02)

# ---------------------------------------------------------------------------
# Title strip
# ---------------------------------------------------------------------------
ax_T = fig.add_subplot(outer[0])
ax_T.set_facecolor(BG); ax_T.axis("off")
ax_T.text(0.5, 0.55,
    "THE GREAT BALANCE-OFF :  PID  vs  LQR  vs  MPC",
    ha="center", va="center", transform=ax_T.transAxes,
    fontsize=20, fontweight="bold", color=WHITE, fontfamily="monospace")
ax_T.text(0.996, 0.55, "Mugambi Ndwiga  |  @craftsandengineering",
    ha="right", va="center", transform=ax_T.transAxes, fontsize=7.5, color=DIM)
ax_T.text(0.004, 0.55, "2x slow motion",
    ha="left", va="center", transform=ax_T.transAxes,
    fontsize=7.5, color=DIM, fontstyle="italic")

# ---------------------------------------------------------------------------
# Arena panels  (rec #3, #22)
# ---------------------------------------------------------------------------
arena_gs = gridspec.GridSpecFromSubplotSpec(
    1, 3, subplot_spec=outer[1], wspace=0.012)

arena_axes = []
for i, ctrl in enumerate(CTRL_NAMES):
    ax = fig.add_subplot(arena_gs[i])
    ax.set_facecolor(PANEL)
    ax.set_xlim(-CAM_HALF, CAM_HALF)
    ax.set_ylim(CAM_Y0, CAM_Y1)
    ax.set_aspect("equal")
    ax.tick_params(left=False, labelleft=False, bottom=False, labelbottom=False)
    for sp in ax.spines.values():
        sp.set_color(COLS[ctrl]); sp.set_linewidth(2.8)
    arena_axes.append(ax)

# ---------------------------------------------------------------------------
# Phase strip  (rec #8, #18, #19)
# ---------------------------------------------------------------------------
ax_ph = fig.add_subplot(outer[2])
ax_ph.set_facecolor(BG); ax_ph.axis("off")
phase_lbl = ax_ph.text(0.5, 0.68, "",
    ha="center", va="center", transform=ax_ph.transAxes,
    fontsize=11, fontweight="bold", color=GOLD)
dist_lbl  = ax_ph.text(0.5, 0.08, "",
    ha="center", va="bottom", transform=ax_ph.transAxes,
    fontsize=8.5, color=DIM2, fontfamily="monospace")

# ---------------------------------------------------------------------------
# Compact telemetry strip  (rec #4)
# ---------------------------------------------------------------------------
tel_gs = gridspec.GridSpecFromSubplotSpec(
    1, 4, subplot_spec=outer[3], wspace=0.52)

def tiny_ax(slot, title, ylabel, ylim, yticks=None):
    ax = fig.add_subplot(slot)
    ax.set_facecolor(INSET)
    ax.set_xlim(0, T_SIM); ax.set_ylim(*ylim)
    ax.set_title(title, color=WHITE, fontsize=6.5, pad=2, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=5.5, color=DIM2, labelpad=2)
    ax.set_xlabel("t  [s]", fontsize=5.5, color=DIM2, labelpad=1)
    ax.tick_params(labelsize=5, colors=DIM2, length=2, pad=1)
    for sp in ax.spines.values(): sp.set_color(BDR); sp.set_linewidth(0.5)
    ax.axhline(0, color=DIM, lw=0.5)
    ax.axvline(DIST_T, color="#993333", lw=0.9, ls="--", alpha=0.85)
    if yticks is not None: ax.set_yticks(yticks)
    return ax

ax_ang = tiny_ax(tel_gs[0], "Pole angle  theta [deg]",
    "deg", (-25, 25), [-20, 0, 20])
ax_pos = tiny_ax(tel_gs[1], "Cart position  x [m]",
    "m", (-5, 5), [-4, 0, 4])
ax_frc = tiny_ax(tel_gs[2], "Control force  F [N]",
    "N", (-65, 65), [-60, 0, 60])
ax_nrg = tiny_ax(tel_gs[3], "Cumul. energy  int(F^2)dt  [J]",
    "J", (0, MAX_NRG))

# legend in angle plot
for ctrl in CTRL_NAMES:
    ax_ang.plot([], [], color=COLS[ctrl], lw=1.2, label=ctrl)
ax_ang.legend(fontsize=5, facecolor="#0a0a1a", labelcolor=WHITE,
              edgecolor=BDR, loc="upper right", framealpha=0.9,
              handlelength=1.2, borderpad=0.4)

# PID drift note  (rec #20)
ax_pos.text(0.97, 0.97,
    "PID: no x feedback\n(drift by design)",
    ha="right", va="top", transform=ax_pos.transAxes,
    fontsize=4.8, color=COLS["PID"], fontfamily="monospace", fontstyle="italic")

# Line artists
ang_lines=[]; pos_lines=[]; frc_lines=[]; nrg_lines=[]
for ctrl in CTRL_NAMES:
    al,=ax_ang.plot([],[],color=COLS[ctrl],lw=1.0,zorder=4)
    pl,=ax_pos.plot([],[],color=COLS[ctrl],lw=1.0,zorder=4)
    fl,=ax_frc.plot([],[],color=COLS[ctrl],lw=1.0,zorder=4)
    nl,=ax_nrg.plot([],[],color=COLS[ctrl],lw=1.0,zorder=4)
    ang_lines.append(al); pos_lines.append(pl)
    frc_lines.append(fl); nrg_lines.append(nl)

cursors=[]
for _a, yl in ((ax_ang,(-25,25)),(ax_pos,(-5,5)),(ax_frc,(-65,65)),(ax_nrg,(0,MAX_NRG))):
    c,=_a.plot([],[],color=WHITE,lw=0.5,alpha=0.3,zorder=3)
    cursors.append((c,yl))

# =============================================================================
# 9.  DRAW SCENE  (camera-follow — rec #1, #11-#17)
# =============================================================================
def draw_scene(ax, x_cart, theta, u_force, t_now, col, ctrl_name):
    """
    All coordinates in camera frame: cart is always at cx=0.
    World coordinates: subtract x_cart.
    """
    cx = 0.0
    ax.set_xlim(-CAM_HALF, CAM_HALF)
    ax.set_ylim(CAM_Y0, CAM_Y1)
    ax.set_facecolor(PANEL)

    # Background grid — scrolls with camera  (rec #13)
    for gx_w in range(int(np.floor(x_cart - CAM_HALF)),
                      int(np.ceil( x_cart + CAM_HALF)) + 1):
        ax.axvline(gx_w - x_cart, color="#10102a", lw=0.7, zorder=0)
    for gy in np.arange(0.0, CAM_Y1, 0.4):
        ax.axhline(gy, color="#10102a", lw=0.5, zorder=0)

    # Rail
    ax.fill_between([-CAM_HALF, CAM_HALF], [-0.048,-0.048], [0,0],
                    color="#1a1a3a", zorder=1)
    ax.plot([-CAM_HALF, CAM_HALF], [0,0], color="#3a3a60", lw=1.8, zorder=2)

    # Scrolling metre markers  (rec #13)
    for xi_w in range(int(np.floor(x_cart-CAM_HALF-1)),
                      int(np.ceil( x_cart+CAM_HALF+1))+1):
        xi_c = xi_w - x_cart
        if -CAM_HALF <= xi_c <= CAM_HALF:
            ax.plot([xi_c, xi_c], [-0.022, 0.022], color=DIM, lw=1.0, zorder=3)
            ax.text(xi_c, -0.08, f"{int(xi_w)}",
                    ha="center", va="top", fontsize=7.5, color=DIM2,
                    fontfamily="monospace", zorder=3)

    # World-x readout  (rec #13)
    ax.text(0.0, -0.20, f"x = {x_cart:+.3f} m",
            ha="center", va="top", fontsize=8.5, color=DIM2,
            fontfamily="monospace", zorder=14)
    ax.text(-CAM_HALF + 0.06, -0.08, "world x [m]",
            ha="left", va="top", fontsize=6, color=DIM,
            fontfamily="monospace", zorder=3)

    # Cart  (rec #14)
    ax.add_patch(FancyBboxPatch(
        (cx-CART_W/2, 0), CART_W, CART_H,
        boxstyle="round,pad=0.018",
        facecolor=col, edgecolor=WHITE, lw=1.5, alpha=0.90, zorder=5))
    ax.text(cx, CART_H*0.50, f"M = {M_CART:.0f} kg",
            ha="center", va="center", fontsize=8.5, color=WHITE,
            alpha=0.70, fontweight="bold", zorder=7)

    # Wheels
    wy = -0.042
    for wx in (cx - CART_W*0.30, cx + CART_W*0.30):
        ax.add_patch(plt.Circle((wx, wy), 0.056, color="#44445a", zorder=4))
        ax.add_patch(plt.Circle((wx, wy), 0.026, color="#0d0d20", zorder=5))
        for sa in (0, np.pi/2, np.pi, 3*np.pi/2):
            r = 0.042
            ax.plot([wx, wx+r*np.cos(sa)],[wy, wy+r*np.sin(sa)],
                    color="#5a5a78", lw=0.9, zorder=5)

    # Hinge
    hx, hy = cx, CART_H/2
    ax.add_patch(plt.Circle((hx, hy), 0.046, color=WHITE,    zorder=9))
    ax.add_patch(plt.Circle((hx, hy), 0.022, color="#0d0d20", zorder=10))

    # Vertical reference
    ax.plot([hx,hx],[hy, hy+POLE_VIS+0.15],
            color=DIM, lw=1.0, ls="--", zorder=4, alpha=0.55)
    ax.text(hx+0.05, hy+POLE_VIS+0.17, "vertical",
            ha="left", va="bottom", fontsize=6.5, color=DIM,
            fontfamily="monospace", alpha=0.65)

    # Pole  (rec #12)
    tip_x = hx + POLE_VIS*np.sin(theta)
    tip_y = hy  + POLE_VIS*np.cos(theta)
    ax.plot([hx,tip_x],[hy,tip_y],
            color=col, lw=16, zorder=5, solid_capstyle="round", alpha=0.10)  # glow
    ax.plot([hx,tip_x],[hy,tip_y],
            color=WHITE, lw=6.5, zorder=6, solid_capstyle="round")
    ax.plot([hx,tip_x],[hy,tip_y],
            color=col,   lw=3.2, zorder=7, solid_capstyle="round", alpha=0.82)
    # pole mass label
    pmx = hx + 0.55*POLE_VIS*np.sin(theta) + 0.07*np.cos(theta)
    pmy = hy + 0.55*POLE_VIS*np.cos(theta) - 0.07*np.sin(theta)
    ax.text(pmx, pmy, f"m={M_POLE:.1f}kg",
            ha="center", va="center", fontsize=7.5, color=WHITE,
            alpha=0.60, zorder=8)

    # Bob
    ax.add_patch(plt.Circle((tip_x,tip_y), 0.084,
                             color=col, ec=WHITE, lw=1.5, zorder=11, alpha=0.94))
    ax.add_patch(plt.Circle((tip_x,tip_y), 0.036,
                             color=WHITE, zorder=12, alpha=0.88))

    # Angle arc + label  (rec #16)
    ang_deg = theta * 180/np.pi
    arc_r   = 0.38
    if abs(ang_deg) > 0.8:
        th1 = min(90.0, 90.0 - ang_deg)
        th2 = max(90.0, 90.0 - ang_deg)
        ax.add_patch(Arc((hx,hy), 2*arc_r, 2*arc_r, angle=0,
                         theta1=th1, theta2=th2,
                         color=GOLD, lw=2.0, zorder=12, alpha=0.90))
        mid_a = np.deg2rad((th1+th2)/2)
        lx = hx + (arc_r+0.13)*np.cos(mid_a)
        ly = hy + (arc_r+0.13)*np.sin(mid_a)
        ax.text(lx, ly, r"$\theta$",
                ha="center", va="center", fontsize=14,
                color=GOLD, fontweight="bold", zorder=13)

    # Force arrow  (rec #15)
    if abs(u_force) > 0.5:
        alen = np.sign(u_force) * float(np.clip(
                   0.12 + abs(u_force)*0.018, 0.12, 0.72))
        ay = CART_H*0.56
        ax.annotate("",
            xy=(cx + alen, ay),
            xytext=(cx - np.sign(alen)*CART_W*0.52, ay),
            arrowprops=dict(arrowstyle="-|>", color="#dd8800",
                            lw=3.0, mutation_scale=20), zorder=14)
        ax.text(cx + alen + np.sign(alen)*0.07, ay+0.07, "F",
                ha="center", va="bottom", fontsize=12,
                color="#dd8800", fontweight="bold", zorder=14)

    # State readout  (rec #14)
    ang_col = RED_C if abs(ang_deg)>20 else AMBER if abs(ang_deg)>8 else GRN
    ro = [(f"theta = {ang_deg:+6.2f} deg", ang_col),
          (f"x     = {x_cart:+6.3f} m",    DIM2),
          (f"F     = {u_force:+6.1f} N",   "#dd8800")]
    ro_x = -CAM_HALF + 0.08
    ro_y = CAM_Y1 - 0.08
    for ri, (txt, c) in enumerate(ro):
        ax.text(ro_x, ro_y - ri*0.22, txt,
                ha="left", va="top", fontsize=11, color=c,
                fontfamily="monospace", fontweight="bold", zorder=15)

    # Controller name + sub-label  (rec #17)
    ax.text(0.50, 0.990, ctrl_name,
            ha="center", va="top", transform=ax.transAxes,
            fontsize=22, fontweight="black", color=col,
            fontfamily="monospace", zorder=15)
    SUBLBL = {"PID": "angle + rate only",
              "LQR": "full-state feedback",
              "MPC": "predictive  (25-step horizon)"}
    ax.text(0.50, 0.950, SUBLBL[ctrl_name],
            ha="center", va="top", transform=ax.transAxes,
            fontsize=8, color=DIM2, fontstyle="italic", zorder=15)

    # Disturbance flash
    win = 0.55
    if DIST_T - DT <= t_now < DIST_T + win:
        fade = max(0.0, 1.0 - (t_now - DIST_T)/win)
        ax.fill_between([-CAM_HALF, CAM_HALF],
                        [CAM_Y0,CAM_Y0],[CAM_Y1,CAM_Y1],
                        color=RED_C, alpha=0.08*fade, zorder=0)
        ax.text(0.50, 0.54, "IMPULSE",
                ha="center", va="center", transform=ax.transAxes,
                fontsize=26, fontweight="black", color=RED_C,
                alpha=fade, zorder=16)
        ax.text(0.50, 0.45,
                f"omega_kick = {DIST_ANGVEL:.1f} rad/s",
                ha="center", va="center", transform=ax.transAxes,
                fontsize=10, color="#cc7777", alpha=fade*0.9,
                fontfamily="monospace", zorder=16)

# =============================================================================
# 10.  RESULTS CARD  (rec #6, #7)
#      Pre-created hidden axes; revealed by set_visible(True).
# =============================================================================
res_ax = fig.add_axes([0.11, 0.05, 0.78, 0.88])
res_ax.set_facecolor("#04040d")
res_ax.set_xlim(0,1); res_ax.set_ylim(0,1)
res_ax.tick_params(left=False, labelleft=False, bottom=False, labelbottom=False)
for sp in res_ax.spines.values():          # (rec #7)
    sp.set_color(COLS[WINNER]); sp.set_linewidth(5)
res_ax.set_visible(False)
res_ax.set_zorder(60)

def _t(x, y, s, **kw):
    res_ax.text(x, y, s, transform=res_ax.transAxes, **kw)

def _line(y):
    res_ax.plot([0.03,0.97],[y,y], transform=res_ax.transAxes,
                color=BDR, lw=0.9)

# Static content — drawn once now, revealed later
_t(0.50, 0.952, "FINAL  RESULTS",
   ha="center", va="center", fontsize=14, fontweight="bold",
   color=DIM2, fontfamily="monospace")
res_ax.plot([0.03,0.97],[0.915,0.915], transform=res_ax.transAxes,
            color=COLS[WINNER], lw=1.5, alpha=0.6)

_t(0.50, 0.845, WINNER,
   ha="center", va="center", fontsize=64, fontweight="black",
   color=COLS[WINNER], fontfamily="monospace")

_t(0.50, 0.782,
   "ranked first on combined score  (peak angle + recovery time + control energy)",
   ha="center", va="center", fontsize=9, color=DIM2, fontstyle="italic")
_line(0.752)

col_x = [0.04, 0.26, 0.48, 0.68]
for xi, h in zip(col_x, ["Controller","Peak theta [deg]","Recovery [s]","Energy [J]"]):
    _t(xi, 0.718, h, ha="left", va="center",
       fontsize=9, color=DIM2, fontweight="bold", fontfamily="monospace")

RANK_STR = {1:"1st", 2:"2nd", 3:"3rd"}
for ri, ctrl in enumerate(CTRL_NAMES):
    y = 0.718 - 0.105*(ri+1)
    c = COLS[ctrl]
    for xi, txt in zip(col_x, [
            ctrl,
            f"{metrics[ctrl]['peak_deg']:7.2f}   [{RANK_STR[rP[ctrl]]}]",
            f"{metrics[ctrl]['recovery_s']:7.3f}   [{RANK_STR[rR[ctrl]]}]",
            f"{metrics[ctrl]['energy']:7.1f}   [{RANK_STR[rE[ctrl]]}]"]):
        _t(xi, y, txt, ha="left", va="center",
           fontsize=11, color=c, fontweight="bold", fontfamily="monospace")

_line(0.378)
_t(0.50, 0.348,
   "Peak theta : max deviation after disturbance     "
   "Recovery : time until |theta| < 1.5 deg     "
   "Energy : integral of F^2 dt",
   ha="center", va="center", fontsize=7.5, color=DIM)
_t(0.50, 0.310,
   "PID uses angle + rate feedback only — cart drifts because there is no position term in the law.",
   ha="center", va="center", fontsize=8, color=DIM2, fontstyle="italic")
_t(0.50, 0.278,
   "LQR and MPC use full-state feedback including cart position and velocity.",
   ha="center", va="center", fontsize=8, color=DIM2, fontstyle="italic")

DESC = {"PID":"Proportional-Integral-Derivative  |  tuned gains  |  no model",
        "LQR":"Linear-Quadratic Regulator  |  Riccati solution  |  linear model",
        "MPC":"Model-Predictive Control  |  QP optimisation  |  25-step horizon"}
for ri, ctrl in enumerate(CTRL_NAMES):
    _t(0.50, 0.235 - ri*0.042,
       f"{ctrl} :  {DESC[ctrl]}",
       ha="center", va="center", fontsize=7.5,
       color=COLS[ctrl], fontfamily="monospace")

_line(0.112)
_t(0.50, 0.077, "Mugambi Ndwiga  |  @craftsandengineering",
   ha="center", va="center", fontsize=12, color=DIM2)
_t(0.50, 0.032,
   "The Great Balance-Off: PID vs. LQR vs. MPC  —  Inverted pendulum on a cart",
   ha="center", va="center", fontsize=7.5, color=DIM, fontstyle="italic")

SHOW_WINNER  = T_SIM - 3.5
winner_drawn = False

# =============================================================================
# 11.  ANIMATION
# =============================================================================
def init():
    for l in ang_lines+pos_lines+frc_lines+nrg_lines: l.set_data([],[])
    for c,_ in cursors: c.set_data([],[])
    res_ax.set_visible(False)
    return ang_lines+pos_lines+frc_lines+nrg_lines+[c for c,_ in cursors]

def animate(frame):
    global winner_drawn
    k   = min(frame * EVERY, N-1)
    t_k = k * DT
    sl  = time_arr[:k+1]

    # Phase strip  (rec #8, #18, #19)
    impl = f"{M_POLE*L_HALF*DIST_ANGVEL:.3f} kg.m^2/s"
    if t_k < DIST_T - 0.12:
        phase_lbl.set_text("Steady-state balancing")
        phase_lbl.set_color(GRN)
        dist_lbl.set_text(
            f"Disturbance at t = {DIST_T:.1f} s :   "
            f"omega_kick = {DIST_ANGVEL:.1f} rad/s   "
            f"(angular momentum impulse = {impl})")
    elif t_k < DIST_T + 0.6:
        phase_lbl.set_text("IMPULSE  DISTURBANCE")
        phase_lbl.set_color(RED_C)
        dist_lbl.set_text(
            f"omega_kick = {DIST_ANGVEL:.1f} rad/s   "
            f"angular momentum = {impl}   "
            f"applied at t = {DIST_T:.1f} s")
    elif t_k < DIST_T + 7.0:
        phase_lbl.set_text("Recovery")
        phase_lbl.set_color(AMBER)
        dist_lbl.set_text(
            f"Disturbance: omega_kick = {DIST_ANGVEL:.1f} rad/s at t = {DIST_T:.1f} s   "
            f"|   PID cart drifts by design (no position term in control law)")
    elif t_k < SHOW_WINNER:
        phase_lbl.set_text("Re-stabilised")
        phase_lbl.set_color("#4ecdc4")
        dist_lbl.set_text("")
    else:
        phase_lbl.set_text(""); dist_lbl.set_text("")

    # Telemetry lines
    for i, ctrl in enumerate(CTRL_NAMES):
        th = SIM[ctrl]["states"][:k+1,2]*180/np.pi
        xc = SIM[ctrl]["states"][:k+1,0]
        fc = SIM[ctrl]["forces"][:k+1]
        ne = cum_nrg[ctrl][:k+1]
        ang_lines[i].set_data(sl,th); pos_lines[i].set_data(sl,xc)
        frc_lines[i].set_data(sl,fc); nrg_lines[i].set_data(sl,ne)

    for c,yl in cursors: c.set_data([t_k,t_k], yl)

    # Arena panels — camera-follow  (rec #1)
    for i, (ctrl, ax) in enumerate(zip(CTRL_NAMES, arena_axes)):
        ax.cla()
        ax.set_facecolor(PANEL)
        ax.tick_params(left=False,labelleft=False,bottom=False,labelbottom=False)
        for sp in ax.spines.values():
            sp.set_color(COLS[ctrl]); sp.set_linewidth(2.8)
        draw_scene(ax,
                   SIM[ctrl]["states"][k,0],
                   SIM[ctrl]["states"][k,2],
                   SIM[ctrl]["forces"][k],
                   t_k, COLS[ctrl], ctrl)

    # Results card  (rec #6)
    if t_k >= SHOW_WINNER and not winner_drawn:
        res_ax.set_visible(True)
        winner_drawn = True

    return (ang_lines+pos_lines+frc_lines+nrg_lines+
            [c for c,_ in cursors]+[phase_lbl,dist_lbl])

# =============================================================================
# 12.  RENDER  (rec #11)
# =============================================================================
N_FRAMES = N // EVERY
print(f"  Slow={SLOW}x  |  T_SIM={T_SIM}s  |  video ~{T_SIM*SLOW:.0f}s  |  frames={N_FRAMES}")

ani = FuncAnimation(fig, animate, frames=N_FRAMES,
                    init_func=init, blit=False, interval=1000/FPS)

OUT = "/mnt/user-data/outputs/pid_lqr_mpc_battle.mp4"
os.makedirs(os.path.dirname(OUT), exist_ok=True)

writer = FFMpegWriter(fps=FPS, bitrate=5500, metadata={
    "title"  : "The Great Balance-Off: PID vs. LQR vs. MPC",
    "artist" : "Mugambi Ndwiga - @craftsandengineering",
    "comment": "Inverted pendulum control benchmark — 2x slow motion",
})

print(f"Rendering {N_FRAMES} frames ...")
ani.save(OUT, writer=writer, dpi=90,      # rec #11
         progress_callback=lambda i,n: (
             open("/tmp/rp.txt","w").write(f"{i}/{n}\n") if i%25==0 else None))
print(f"\nSaved -> {OUT}")

# =============================================================================
# 13.  COLAB SNIPPET  (rec #23)
# =============================================================================

from IPython.display import HTML, display
from google.colab import files
import base64

VIDEO = OUT  # Use the absolute path defined above

with open(VIDEO, "rb") as fh:
    b64 = base64.b64encode(fh.read()).decode()

display(HTML(f"""
<div style="background:#07071a;padding:14px;border-radius:8px;
            border:2px solid #1c1c3c;display:inline-block;">
  <p style="color:#dcdcec;font-family:monospace;font-size:14px;
             text-align:center;margin:0 0 8px 0;">
    THE GREAT BALANCE-OFF : PID vs LQR vs MPC &nbsp;|&nbsp; 2x slow motion
  </p>
  <video width="1280" height="720" controls autoplay loop
         style="border-radius:4px;display:block;">
    <source src="data:video/mp4;base64,{b64}" type="video/mp4">
  </video>
  <p style="color:#6a6a9a;font-family:monospace;font-size:11px;
             text-align:center;margin:6px 0 0 0;">
    Mugambi Ndwiga &nbsp;|&nbsp; @craftsandengineering
  </p>
</div>
"""))

files.download(VIDEO)


Simulating ...
  Metrics : {'PID': {'peak_deg': 7.9987882761547695, 'energy': 143.3543239733911, 'recovery_s': 0.23500000000000001}, 'LQR': {'peak_deg': 15.481115075458808, 'energy': 136.27757781241516, 'recovery_s': 2.2}, 'MPC': {'peak_deg': 11.055967300943195, 'energy': 107.8286753416479, 'recovery_s': 3.325}}
  Winner  : PID  scores={'PID': 5, 'LQR': 7, 'MPC': 6}
  Slow=2x  |  T_SIM=12.0s  |  video ~24s  |  frames=800
Rendering 800 frames ...

Saved -> /mnt/user-data/outputs/pid_lqr_mpc_battle.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>